In [0]:
%pip install websockets
%pip install nest_asyncio

In [0]:
import asyncio
import nest_asyncio
import websockets
import json
import os
import time
from datetime import datetime, timezone

nest_asyncio.apply()

async def connect_ais_stream():
    url = "wss://stream.aisstream.io/v0/stream"
    apiKey = "f244d8bb87bad16d75a7dd76b394f7e8ccac3f5a"

    CATALOG = "workspace" 
    SCHEMA = "ais_db"
    VOLUME = "ais_raw_data"
    FOLDERNAME = "bronze_ais"
    base_volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FOLDERNAME}"
    if not os.path.exists(base_volume_path):
        os.makedirs(base_volume_path, exist_ok=True)

    buffer = []
    write_interval = 10 # Save every 10 seconds
    last_write = time.time()

    try:
        async with websockets.connect(url) as websocket:
            subscribe_message = {"APIKey": apiKey,
                                "BoundingBoxes": [[[53.45, 9.80], [53.60, 10.05]]]
                                }
            
            subscribe_message_json = json.dumps(subscribe_message)
            await websocket.send(subscribe_message_json)
            print(f"📡 Connection established. Writing to {base_volume_path}")
           
            async for message_raw in websocket:
                message_json = message_raw.decode("utf-8")
                buffer.append(message_json)

                current_time = time.time()
                elapsed_time = current_time - last_write
                print(elapsed_time)
                if elapsed_time >= write_interval:
                    if buffer:
                        print("Starting to write!")
                        batch_data = "\n".join(buffer)
                        
                        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                        file_path = f"{base_volume_path}/batch_{ts}.jsonl"

                        with open(file_path, "w") as f:
                            f.write(batch_data)
                            print(f"✅ Wrote {len(buffer)} messages to {file_path}")
                        buffer = []
                        last_write = current_time
                           
    except websockets.exceptions.ConnectionClosedError:
        print("❌ Connection lost. The server closed the connection.")
    except Exception as e:
        print(f"🔍 An unexpected error occurred: {e}")
    

if __name__ == "__main__":
    asyncio.run(connect_ais_stream())
      





In [0]:
# 1. Setup your Path Variables
CATALOG = "workspace" 
SCHEMA = "ais_db"
VOLUME = "ais_raw_data"
FOLDERNAME = "bronze_ais"

# The source where your Python script drops JSON files
base_volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FOLDERNAME}/"

# The "Memory" folder for Spark (Checkpointing)
# We put this in a separate folder inside the Volume
checkpoint_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoints/bronze_ais/"

# The final destination (Table Name)
target_table = f"{CATALOG}.{SCHEMA}.bronze_ais_table"

# 2. Configure the Auto Loader Stream
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")                   # Read your raw JSON strings
    .option("cloudFiles.schemaLocation", checkpoint_path)  # Store the table structure here
    .option("cloudFiles.inferColumnTypes", "true")         # Automatically detect numbers vs strings
    .load(base_volume_path))

# 3. Execute the Write to Delta Lake
query = (df_stream.writeStream
    .option("checkpointLocation", checkpoint_path)         # Remember which files were already processed
    .option("mergeSchema", "true")                         # Allow table to grow if new AIS fields appear
    .trigger(availableNow=True)                            # Process all new files and stop (Batch mode)
    .toTable(target_table))

# 4. Wait for the process to finish
query.awaitTermination()

print(f"✅ Successfully updated {target_table} from {base_volume_path}")

In [0]:
display(spark.table(f"workspace.ais_db.bronze_ais_table"))